# Planner model performance

Compare language models from one `plannerbench` run. Keeping the comparison inside a single run ensures every model saw the same exported backend prompt, cases, repeats, concurrency, and provider conditions.

By default this notebook opens the planner manifest referenced by `reports/latest.json`. Set `BIOCOSMOS_PLANNER_MANIFEST` to a specific `run.json` before starting Jupyter to reproduce an older run. Token counts are provider-reported; per-call values use successful responses only.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "backend/app/configs/config.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
import math
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.lines import Line2D

from analyses.helpers.publication import (
    DEFAULT_PALETTE,
    export_figure,
    load_settings,
    publication_style,
)

settings = load_settings(ROOT)
publication_style()
plt.rcParams.update(
    {
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.labelsize": 13,
        "xtick.labelsize": 11.5,
        "ytick.labelsize": 11.5,
        "legend.fontsize": 11.5,
        "legend.title_fontsize": 12.5,
        "figure.titlesize": 16,
    }
)

## Load one benchmark run

The manifest contains aggregate model metrics; `trials.jsonl` contains one record per model, case, and repeat. Absolute artifact paths recorded on another machine are intentionally ignored—the trials file is resolved beside the selected manifest.

In [ ]:
manifest_override = os.getenv("BIOCOSMOS_PLANNER_MANIFEST")
if manifest_override:
    manifest_path = Path(manifest_override).expanduser().resolve()
else:
    reports_dir = ROOT / "reports"
    latest_path = reports_dir / "latest.json"
    if not latest_path.is_file():
        raise FileNotFoundError(
            "No reports/latest.json found. Run plannerbench or set "
            "BIOCOSMOS_PLANNER_MANIFEST to a planner run.json."
        )
    latest = json.loads(latest_path.read_text(encoding="utf-8"))
    planner_latest = latest.get("planner")
    if not planner_latest or not planner_latest.get("manifest"):
        raise ValueError("reports/latest.json has no planner benchmark entry")
    manifest_path = (reports_dir / planner_latest["manifest"]).resolve()

if not manifest_path.is_file():
    raise FileNotFoundError(f"Planner manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
trials_path = manifest_path.parent / "trials.jsonl"
if not trials_path.is_file():
    raise FileNotFoundError(f"Planner trials not found beside manifest: {trials_path}")
trials = pd.read_json(trials_path, lines=True)

models = manifest.get("models", [])
if not models or not manifest.get("summaries"):
    raise ValueError("Planner manifest has no models or summaries")
if trials.empty:
    raise ValueError("Planner trials file is empty")
model_palette = dict(
    zip(models, sns.color_palette(DEFAULT_PALETTE, n_colors=len(models)), strict=True)
)

print(f"Run: {manifest.get('run_id', manifest_path.parent.name)}")
print(f"Production model: {manifest.get('production_model', 'unknown')}")
print(f"Prompt fingerprint: {manifest.get('spec_fingerprint', 'unknown')}")
print(
    f"{len(manifest.get('case_ids', []))} cases × {manifest.get('repeats', '?')} repeats "
    f"× {len(models)} models = {len(trials)} recorded trials"
)

## Overall comparison

Accuracy includes API errors as misses. Consistency is the share of cases whose repeats all succeeded with the same accepted plan. Token totals cover successful calls; the normalized column makes models comparable when their error counts differ.

In [ ]:
summary = pd.DataFrame(manifest["summaries"]).set_index("model").reindex(models)
unavailable_models = summary.index[summary["api_errors"] == summary["trials"]].tolist()
if unavailable_models:
    print(
        "WARNING: no successful calls for: "
        + ", ".join(unavailable_models)
        + ". Treat these as access/provider failures, not model-performance results."
    )
for column in ("prompt_tokens", "completion_tokens"):
    if column not in summary:
        summary[column] = 0
if "total_tokens" not in summary:
    summary["total_tokens"] = summary["prompt_tokens"] + summary["completion_tokens"]
summary["consistent_rate"] = summary["consistent_cases"] / summary["case_count"]
summary["error_rate"] = summary["api_errors"] / summary["trials"]
summary["successful_trials"] = summary["trials"] - summary["api_errors"]
summary["tokens_per_success"] = summary["total_tokens"] / summary["successful_trials"].replace(
    0, pd.NA
)

comparison = summary[
    [
        "accuracy",
        "consistent_rate",
        "no_tool_rate",
        "invalid_call_rate",
        "error_rate",
        "latency_p50_seconds",
        "latency_p95_seconds",
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
        "tokens_per_success",
    ]
]
display(
    comparison.style.format(
        {
            "accuracy": "{:.1%}",
            "consistent_rate": "{:.1%}",
            "no_tool_rate": "{:.1%}",
            "invalid_call_rate": "{:.1%}",
            "error_rate": "{:.1%}",
            "latency_p50_seconds": "{:.3f}",
            "latency_p95_seconds": "{:.3f}",
            "prompt_tokens": "{:,.0f}",
            "completion_tokens": "{:,.0f}",
            "total_tokens": "{:,.0f}",
            "tokens_per_success": "{:,.1f}",
        }
    )
)

In [ ]:
metric_colors = sns.color_palette(DEFAULT_PALETTE, n_colors=3)
fig_height = max(10, len(models) * 1.35)
fig, axes = plt.subplots(2, 2, figsize=(16, fig_height), layout="constrained")

summary[["accuracy", "consistent_rate"]].plot.barh(ax=axes[0, 0], color=metric_colors[:2])
axes[0, 0].set(title="Planner quality", xlabel="Share", ylabel="", xlim=(0, 1))
axes[0, 0].legend(
    ["Accuracy", "Consistent cases"], loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False
)

summary[["latency_p50_seconds", "latency_p95_seconds"]].plot.barh(
    ax=axes[0, 1], color=metric_colors[:2]
)
axes[0, 1].set(title="Successful-call latency", xlabel="Seconds", ylabel="")
axes[0, 1].legend(["p50", "p95"], loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)

summary[["prompt_tokens", "completion_tokens"]].plot.barh(
    stacked=True, ax=axes[1, 0], color=metric_colors[:2]
)
axes[1, 0].set(title="Provider-reported token usage", xlabel="Tokens", ylabel="")
axes[1, 0].legend(
    ["Prompt", "Completion"], loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False
)

summary[["no_tool_rate", "invalid_call_rate", "error_rate"]].plot.barh(
    ax=axes[1, 1], color=metric_colors
)
axes[1, 1].set(title="Failure indicators", xlabel="Share", ylabel="", xlim=(0, 1))
axes[1, 1].legend(
    ["No valid tool", "Invalid call", "API error"],
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=False,
)

for ax in axes.flat:
    sns.despine(ax=ax)
export_figure(fig, settings, "planner_overall_comparison", {"models": comparison.reset_index()})
plt.show()
plt.close(fig)

## Quality–cost trade-offs

Each panel highlights its own Pareto leaders. A model is dominated when another model is at least as accurate and no more costly, with a strict improvement in at least one of those dimensions. Models without successful calls or a finite cost measurement are excluded.

In [ ]:
def pareto_leaders(frame, cost_column):
    leaders = []
    for model, row in frame.iterrows():
        no_worse = (frame["accuracy"] >= row["accuracy"]) & (frame[cost_column] <= row[cost_column])
        strictly_better = (frame["accuracy"] > row["accuracy"]) | (
            frame[cost_column] < row[cost_column]
        )
        if not (no_worse & strictly_better).any():
            leaders.append(model)
    return leaders


scatter_metrics = (
    ("latency_p50_seconds", "p50 latency (seconds)", "Accuracy vs latency"),
    ("tokens_per_success", "Tokens per successful call", "Accuracy vs token usage"),
)
scatter_source = summary.loc[summary["successful_trials"] > 0]


def plot_quality_cost(ax, cost_column, xlabel, title, title_loc="center"):
    panel = (
        scatter_source[["accuracy", cost_column]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
    )
    panel = panel.loc[
        panel["accuracy"].map(math.isfinite) & panel[cost_column].map(math.isfinite)
    ]
    leaders = pareto_leaders(panel, cost_column)
    frontier = panel.loc[leaders].sort_values(cost_column)
    if len(frontier) > 1:
        ax.plot(
            frontier[cost_column],
            frontier["accuracy"],
            color="#333333",
            linewidth=1.25,
            alpha=0.45,
            zorder=1,
        )
    for model, row in panel.iterrows():
        is_leader = model in leaders
        ax.scatter(
            row[cost_column],
            row["accuracy"],
            color=model_palette[model],
            edgecolor="black" if is_leader else "white",
            linewidth=1.6 if is_leader else 0.8,
            s=150 if is_leader else 75,
            alpha=1 if is_leader else 0.72,
            zorder=3 if is_leader else 2,
        )
    if panel.empty:
        ax.text(0.5, 0.5, "No eligible models", transform=ax.transAxes, ha="center")
    ax.set(xlabel=xlabel, ylabel="Accuracy", ylim=(0, 1.03))
    ax.set_title(title, loc=title_loc)
    sns.despine(ax=ax)


legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor=model_palette[model],
        markeredgecolor="white",
        markersize=8,
        label=model,
    )
    for model in models
]
legend_handles.append(
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=1.6,
        markersize=10,
        label="Best performance model",
    )
)

fig, (*axes, legend_ax) = plt.subplots(
    1,
    3,
    figsize=(16, 5.5),
    gridspec_kw={"width_ratios": (1, 1, 0.55)},
    layout="constrained",
)
for ax, (cost_column, xlabel, title) in zip(axes, scatter_metrics, strict=True):
    plot_quality_cost(ax, cost_column, xlabel, title)
legend_ax.axis("off")
legend_ax.legend(handles=legend_handles, title="Model", loc="center left", frameon=False)
fig.suptitle("Planner quality–cost trade-offs", fontweight="bold")
trade_off_data = scatter_source[
    ["accuracy", "latency_p50_seconds", "tokens_per_success", "successful_trials"]
].reset_index()
export_figure(fig, settings, "planner_quality_cost", {"models": trade_off_data})
plt.show()
plt.close(fig)

## Per-case accuracy

The heatmap exposes where aggregate accuracy hides model-specific strengths or weaknesses. Values are correct repeats divided by configured repeats.

In [ ]:
repeats = manifest["repeats"]
case_ids = manifest["case_ids"]
per_case = pd.DataFrame(
    {
        model: {
            case_id: summary.loc[model, "per_case_correct"][case_id] / repeats
            for case_id in case_ids
        }
        for model in models
    }
).T

fig_width = max(12, len(case_ids) * 0.9)
fig_height = max(4, len(models) * 0.7)
accuracy_cmap = sns.light_palette(model_palette[models[0]], as_cmap=True)


def plot_per_case_accuracy(ax, title, title_loc="center"):
    sns.heatmap(
        per_case,
        annot=True,
        fmt=".0%",
        cmap=accuracy_cmap,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Correct repeats"},
        ax=ax,
    )
    ax.set(xlabel="Case", ylabel="Model")
    ax.set_title(title, loc=title_loc)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    ax.tick_params(axis="y", rotation=0)


fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout="constrained")
plot_per_case_accuracy(ax, "Accuracy by benchmark case")
per_case_data = (
    per_case.rename_axis("model")
    .reset_index()
    .melt(id_vars="model", var_name="case", value_name="accuracy")
)
export_figure(fig, settings, "planner_per_case_accuracy", {"cases": per_case_data})
plt.show()
plt.close(fig)

## Combined quality–cost and per-case accuracy

The publication figure reuses the trade-off panels and per-case heatmap above in one two-row layout.

In [ ]:
combined_height = max(10, 5.5 + fig_height)
fig = plt.figure(figsize=(max(16, fig_width), combined_height), layout="constrained")
top, bottom = fig.subfigures(2, 1, height_ratios=(5.5, fig_height))
top_axes = top.subplots(1, 3, gridspec_kw={"width_ratios": (1, 1, 0.55)})
for ax, (cost_column, xlabel, title) in zip(
    top_axes[:2],
    (
        ("latency_p50_seconds", "p50 latency (seconds)", "A) Accuracy vs. latency"),
        ("tokens_per_success", "Tokens per successful call", "B) Accuracy vs. token usage"),
    ),
    strict=True,
):
    plot_quality_cost(ax, cost_column, xlabel, title, title_loc="left")
top_axes[2].axis("off")
top_axes[2].legend(
    handles=legend_handles, title="Model", loc="center left", frameon=False
)
bottom_ax = bottom.subplots()
plot_per_case_accuracy(bottom_ax, "C) Accuracy by test case", title_loc="left")
export_figure(
    fig,
    settings,
    "planner_quality_cost_per_case",
    {"quality_cost": trade_off_data, "per_case": per_case_data},
)
plt.show()
plt.close(fig)

## Trial-level latency and tokens

Distributions reveal outliers that percentiles and aggregate token totals can conceal. Failed calls are excluded because they do not have comparable response usage.

In [ ]:
if "total_tokens" not in trials:
    trials["total_tokens"] = pd.NA
component_total = trials.get("prompt_tokens", 0).fillna(0) + trials.get(
    "completion_tokens", 0
).fillna(0)
trials["total_tokens"] = trials["total_tokens"].fillna(component_total)
successful = trials.loc[trials["status"] == "ok"].copy()

fig_height = max(5, len(models) * 0.75)
fig, axes = plt.subplots(1, 2, figsize=(15, fig_height), layout="constrained")
sns.boxplot(
    data=successful,
    x="latency_seconds",
    y="model",
    order=models,
    hue="model",
    palette=model_palette,
    legend=False,
    ax=axes[0],
)
axes[0].set(title="Latency distribution", xlabel="Seconds", ylabel="")
sns.boxplot(
    data=successful,
    x="total_tokens",
    y="model",
    order=models,
    hue="model",
    palette=model_palette,
    legend=False,
    ax=axes[1],
)
axes[1].set(title="Tokens per successful call", xlabel="Tokens", ylabel="")
for ax in axes:
    sns.despine(ax=ax)
trial_data = successful[["model", "case_id", "latency_seconds", "total_tokens"]]
export_figure(fig, settings, "planner_trial_distributions", {"trials": trial_data})
plt.show()
plt.close(fig)

## Interpretation checklist

- Prefer accuracy and per-case behavior over latency alone; a fast wrong plan produces wrong search results.
- Exclude models with no successful calls from quality comparisons; 100% API errors usually indicate access or provider failure, not model capability.
- Treat consistency as complementary to accuracy: a consistently wrong model is still wrong.
- Compare normalized tokens per successful call when API error counts differ.
- Do not compare separate runs unless their prompt fingerprint, cases, repeats, concurrency, temperature, and endpoint conditions match.
- Re-run the benchmark before changing production models; this notebook analyzes recorded evidence and does not deploy a model.